In [1]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
from ultralytics import YOLO

model = YOLO('yolo26m.pt')


results = model.train(data = r"C:\Users\Heath Dingus\Downloads\My First Project.yolov8\data.yaml", epochs=100, imgsz=1024, device = 0, workers = 0, batch = 4)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.39  Python-3.13.9 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Heath Dingus\Downloads\My First Project.yolov8\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, moment

In [9]:
import cv2
import numpy as np
from ultralytics import YOLO
from cv2_enumerate_cameras import enumerate_cameras
import sys

# 1. Model Initialization
MODEL_PATH = r"C:\Users\Heath Dingus\runs\detect\train-37\weights\best.pt"
model = YOLO(MODEL_PATH)

# 2. Camera Selection Logic
def select_camera():
    s = set()
    u = []
    for camera in enumerate_cameras():
        if camera.name not in s:
            print(f'Index: {camera.index} | Name: {camera.name}') 
            s.add(camera.name)
            u.append(camera.index)
    return int(input('Select camera index: '))

cam_idx = select_camera()
cap = cv2.VideoCapture(cam_idx)

# 3. Scoring Logic
def get_score(distance):
    if distance < 0.03: 
        return 10  
    if distance < 0.07: 
        return 9
    if distance < 0.09: 
        return 8   
    if distance < 0.11: 
        return 7
    if distance < 0.15: 
        return 6   
    if distance < 0.20: 
        return 5
    if distance < 0.25: 
        return 4
    if distance < 0.3: 
        return 3
    else:
        return "Out"

# 4. State Variables
cal_x, cal_y = [], []
final_center = None
CALIBRATION_LIMIT = 100

# Setup Window
cv2.namedWindow('LaneAssist', cv2.WINDOW_NORMAL)

while cap.isOpened():
    ret, frame = cap.read()

    results = model(frame, stream=False, imgsz=[1024, 1024])
    r = results[0]
    annotated_frame = r.plot()
    
    current_center = None
    arrow_data = []

    for box in r.boxes:
        coords = box.xywhn[0].tolist() 
        cls = int(box.cls[0])
        if cls == 1: # Center Ring
            current_center = (coords[0], coords[1])
        elif cls == 0: # Arrow
            arrow_data.append((coords[0], coords[1]))

    if len(cal_x) < CALIBRATION_LIMIT:
        if current_center:
            cal_x.append(current_center[0])
            cal_y.append(current_center[1])
        
        status_color = (0, 255, 255)
        cv2.putText(annotated_frame, f'CALIBRATING: {len(cal_x)}/{CALIBRATION_LIMIT}', 
                    (15, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
    else:
        if final_center is None:
            final_center = (np.mean(cal_x), np.mean(cal_y))
            print(f'Calibration Success! Center Locked at: {final_center}')
        
        cv2.putText(annotated_frame, "CALIBRATED", (15, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    active_center = final_center if final_center else current_center
    if active_center:
        for arrow_xy in arrow_data:
            dist = np.sqrt((arrow_xy[0] - active_center[0])**2 + (arrow_xy[1] - active_center[1])**2)
            score = get_score(dist)
            
            pix_x = int(arrow_xy[0] * frame.shape[1])
            pix_y = int(arrow_xy[1] * frame.shape[0])
            
            cv2.putText(annotated_frame, f'Score: {score}', (pix_x, pix_y - 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.putText(annotated_frame, "LaneAssist v0.1.2", (15, 25), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    
    cv2.imshow('LaneAssist', annotated_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Name: e2eSoft iVCam
 Index: 1400
Name: DroidCam Video
 Index: 1401


Select a camera index:  1401



WARNING imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
0: 1088x1472 (no detections), 36.3ms
Speed: 5.6ms preprocess, 36.3ms inference, 0.3ms postprocess per image at shape (1, 3, 1088, 1472)

WARNING imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
0: 1088x1472 1 Arrow, 1 Center ring, 1 Target, 40.5ms
Speed: 6.4ms preprocess, 40.5ms inference, 0.4ms postprocess per image at shape (1, 3, 1088, 1472)

WARNING imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
0: 1088x1472 1 Arrow, 1 Center ring, 1 Target, 42.6ms
Speed: 7.2ms preprocess, 42.6ms inference, 0.4ms postprocess per image at shape (1, 3, 1088, 1472)

WARNING imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
0: 1088x1472 1 Arrow, 1 Center ring, 1 Target, 43.0ms
Speed: 7.3ms preprocess, 43.0ms inference, 0.4ms postprocess per image at shape (1, 3, 1088, 1472)

WARNING imgsz=[1080, 1920] must be multiple of max s